# PRMP on rel-hm: Predictive Residual Message Passing for Heterogeneous Graphs

This notebook demonstrates the **Predictive Residual Message Passing (PRMP)** experiment on the H&M Fashion relational dataset (rel-hm).

**What it does:**
- Generates a synthetic heterogeneous graph with 3 node types (customer, article, transaction) and 2 FK edge types
- Compares 3 GNN variants: **Standard** HeteroGNN, **PRMP** (with predictive residual convolutions), and **Wide** (parameter-matched control)
- Trains on 2 tasks: user-churn (classification, AUROC) and item-sales (regression, MAE)
- Computes Ridge R² analysis for FK links
- Compares demo results against full-scale reference results

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('torch-geometric==2.6.1')
_pip('torch-scatter', 'torch-sparse', '-f', 'https://data.pyg.org/whl/torch-2.9.0+cpu.html')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')
    _pip('torch==2.9.0+cpu', '-f', 'https://download.pytorch.org/whl/torch_stable.html')

In [ ]:
import gc
import json
import copy
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, HeteroConv
from torch_geometric.utils import scatter

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, roc_auc_score, mean_absolute_error

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-b2d5b0-predictive-residual-message-passing-filt/main/experiment_iter7_prmp_on_rel_hm/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded reference data with keys: {list(data.keys())}")
print(f"Reference results from full experiment:")
ref_meta = data["metadata"]
for task_name, task_res in ref_meta["results_by_task"].items():
    print(f"  {task_name} ({task_res['metric']}):")
    for variant in ["standard", "prmp", "wide"]:
        vr = task_res[variant]
        print(f"    {variant}: {vr['mean']:.4f} +/- {vr['std']:.4f}")

## Configuration

All tunable parameters for the experiment. Demo uses small values for fast execution; original full-scale values are shown in comments.

In [ ]:
# --- Demo config (minimum values for fast execution) ---
HIDDEN_DIM = 32        # Original: 128
NUM_LAYERS = 2         # Original: 2
LR = 0.001             # Original: 0.001
EPOCHS = 10            # Original: 200
PATIENCE = 10          # Original: 20
SEEDS = [42]           # Original: [42, 123, 456, 789, 1024]
VARIANTS = ["standard", "prmp", "wide"]
BATCH_SIZE = 128       # Original: 1024

# Synthetic data sizes (demo)
N_CUSTOMERS = 100      # Original: ~50000
N_ARTICLES = 50        # Original: ~25000
N_TRANSACTIONS = 500   # Original: 500000

TASKS_CFG = {
    "user-churn": {
        "entity": "customer",
        "metric_name": "AUROC",
        "task_type": "classification",
    },
    "item-sales": {
        "entity": "article",
        "metric_name": "MAE",
        "task_type": "regression",
    },
}

## Phase 1: Synthetic Data Generation & Graph Construction

We generate a synthetic relational dataset mimicking the H&M schema with 3 tables (customer, article, transaction) connected by foreign keys. Transactions link customers to articles with realistic power-law cardinality distributions.

In [ ]:
def generate_synthetic_tables(n_customers, n_articles, n_transactions, seed=42):
    """Generate synthetic relational tables mimicking H&M schema."""
    rng = np.random.RandomState(seed)

    # Customer features (5 numeric features)
    customers = pd.DataFrame({
        "customer_id": range(n_customers),
        "age": rng.normal(40, 15, n_customers).clip(15, 90).astype(int),
        "FN": rng.choice([0, 1], n_customers, p=[0.4, 0.6]),
        "Active": rng.choice([0, 1], n_customers, p=[0.3, 0.7]),
        "club_member_status": rng.choice([0, 1, 2], n_customers),
        "fashion_news_frequency": rng.choice([0, 1, 2], n_customers),
    })

    # Article features (5 numeric features)
    articles = pd.DataFrame({
        "article_id": range(n_articles),
        "product_type_no": rng.randint(0, 50, n_articles),
        "product_group_name": rng.randint(0, 10, n_articles),
        "graphical_appearance_no": rng.randint(0, 20, n_articles),
        "colour_group_code": rng.randint(0, 15, n_articles),
        "section_no": rng.randint(0, 30, n_articles),
    })

    # Transactions with power-law FK distribution
    cust_probs = rng.pareto(1.5, n_customers) + 1
    cust_probs /= cust_probs.sum()
    art_probs = rng.pareto(1.0, n_articles) + 1
    art_probs /= art_probs.sum()

    txn_customer_ids = rng.choice(n_customers, size=n_transactions, p=cust_probs)
    txn_article_ids = rng.choice(n_articles, size=n_transactions, p=art_probs)

    # Temporal dates spanning 2018-09 to 2020-09
    base_ts = 1535760000  # 2018-09-01
    end_ts = 1600560000   # 2020-09-20
    txn_dates = pd.to_datetime(rng.randint(base_ts, end_ts, size=n_transactions), unit="s")

    transactions = pd.DataFrame({
        "customer_id": txn_customer_ids,
        "article_id": txn_article_ids,
        "t_dat": txn_dates,
        "price": rng.exponential(0.05, n_transactions).clip(0.001, 1.0),
        "sales_channel_id": rng.choice([1, 2], n_transactions),
    })
    transactions = transactions.sort_values("t_dat").reset_index(drop=True)

    print(f"Generated: customers={len(customers)}, articles={len(articles)}, "
          f"transactions={len(transactions)}")
    return customers, articles, transactions

customers, articles, transactions = generate_synthetic_tables(
    N_CUSTOMERS, N_ARTICLES, N_TRANSACTIONS
)

In [ ]:
def build_hetero_graph(customers, articles, transactions):
    """Build HeteroData graph from relational tables."""
    cust_id_col = "customer_id"
    art_id_col = "article_id"
    date_col = "t_dat"

    # Build node ID mappings
    unique_customers = transactions[cust_id_col].unique()
    unique_articles = transactions[art_id_col].unique()
    cust_map = {cid: i for i, cid in enumerate(unique_customers)}
    art_map = {aid: i for i, aid in enumerate(unique_articles)}
    n_cust = len(cust_map)
    n_art = len(art_map)
    n_txn = len(transactions)

    num_nodes_dict = {"customer": n_cust, "article": n_art, "transaction": n_txn}

    # Build edge indices (bidirectional FK edges)
    txn_cust_src = torch.arange(n_txn, dtype=torch.long)
    txn_cust_dst = torch.tensor([cust_map[c] for c in transactions[cust_id_col]], dtype=torch.long)
    txn_art_src = torch.arange(n_txn, dtype=torch.long)
    txn_art_dst = torch.tensor([art_map[a] for a in transactions[art_id_col]], dtype=torch.long)

    data = HeteroData()
    for ntype, n in num_nodes_dict.items():
        data[ntype].num_nodes = n

    data["transaction", "fk_to", "customer"].edge_index = torch.stack([txn_cust_src, txn_cust_dst])
    data["customer", "rev_fk", "transaction"].edge_index = torch.stack([txn_cust_dst, txn_cust_src])
    data["transaction", "fk_to", "article"].edge_index = torch.stack([txn_art_src, txn_art_dst])
    data["article", "rev_fk", "transaction"].edge_index = torch.stack([txn_art_dst, txn_art_src])

    # Cardinality stats
    cust_card = transactions.groupby(cust_id_col).size()
    art_card = transactions.groupby(art_id_col).size()
    print(f"Graph: {n_cust} customers, {n_art} articles, {n_txn} transactions")
    print(f"Cardinality customer->txn: mean={cust_card.mean():.1f}, max={cust_card.max()}")
    print(f"Cardinality article->txn: mean={art_card.mean():.1f}, max={art_card.max()}")

    # Temporal split (70/15/15)
    dates = transactions[date_col]
    sorted_dates = dates.sort_values()
    val_ts = sorted_dates.iloc[int(len(sorted_dates) * 0.70)]
    test_ts = sorted_dates.iloc[int(len(sorted_dates) * 0.85)]

    train_mask = dates < val_ts
    val_mask = (dates >= val_ts) & (dates < test_ts)
    test_mask = dates >= test_ts
    train_txns = transactions[train_mask]
    val_txns = transactions[val_mask]
    test_txns = transactions[test_mask]

    # User-churn labels
    train_custs = set(train_txns[cust_id_col].unique())
    val_custs = set(val_txns[cust_id_col].unique())
    test_custs = set(test_txns[cust_id_col].unique())

    churn_labels_train = {c: (1.0 if c in val_custs else 0.0) for c in train_custs}
    churn_labels_val = {c: (1.0 if c in test_custs else 0.0) for c in val_custs}
    remaining = train_custs - val_custs - test_custs
    churn_labels_test = {c: 1.0 for c in test_custs}
    for c in list(remaining)[:len(test_custs)]:
        churn_labels_test[c] = 0.0

    # Item-sales labels
    sales_train = train_txns.groupby(art_id_col).size()
    sales_val = val_txns.groupby(art_id_col).size()
    sales_test = test_txns.groupby(art_id_col).size()
    def _make_sales_dict(series):
        return {k: float(np.log1p(v)) for k, v in series.items()}

    task_labels = {
        "user-churn": {"train": churn_labels_train, "val": churn_labels_val, "test": churn_labels_test},
        "item-sales": {"train": _make_sales_dict(sales_train), "val": _make_sales_dict(sales_val),
                        "test": _make_sales_dict(sales_test)},
    }

    node_maps = {
        "customer": cust_map, "article": art_map, "num_nodes": num_nodes_dict,
        "cust_cardinality_mean": float(cust_card.mean()),
        "art_cardinality_mean": float(art_card.mean()),
    }

    return data, node_maps, task_labels

hetero_data, node_maps, task_labels = build_hetero_graph(customers, articles, transactions)
del customers, articles, transactions
gc.collect()

## Phase 2: Model Definitions

Three GNN variants:
1. **StandardHeteroGNN**: Uses SAGEConv for all edge types
2. **PRMPHeteroGNN**: Uses Predictive Residual Message Passing for FK edges (parent->child), SAGEConv for reverse edges
3. **Wide** control: Standard GNN with hidden dim increased to match PRMP's non-embedding parameter count

The PRMP convolution predicts child embeddings from parent, computes residuals, and aggregates them back to update parent representations.

In [ ]:
class StandardHeteroGNN(nn.Module):
    """Standard heterogeneous GNN using SAGEConv per edge type."""

    def __init__(self, num_nodes_dict, metadata, hidden=128, num_layers=2):
        super().__init__()
        self.hidden = hidden
        self.embeddings = nn.ModuleDict({
            ntype: nn.Embedding(n, hidden) for ntype, n in num_nodes_dict.items()
        })
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            conv_dict = {}
            for edge_type in metadata[1]:
                conv_dict[edge_type] = SAGEConv(hidden, hidden)
            self.convs.append(HeteroConv(conv_dict, aggr="sum"))
        self.norms = nn.ModuleList([
            nn.ModuleDict({ntype: nn.LayerNorm(hidden) for ntype in num_nodes_dict})
            for _ in range(num_layers)
        ])
        self.head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, 1))

    def get_embeddings(self):
        return {ntype: emb.weight for ntype, emb in self.embeddings.items()}

    def forward(self, edge_index_dict, target_type, target_ids):
        x_dict = self.get_embeddings()
        for i, conv in enumerate(self.convs):
            x_dict_new = conv(x_dict, edge_index_dict)
            x_dict = {k: F.relu(self.norms[i][k](x_dict_new.get(k, x_dict[k]))) for k in x_dict}
        return self.head(x_dict[target_type][target_ids]).squeeze(-1)


class PRMPConv(nn.Module):
    """Predictive Residual Message Passing convolution.

    For each FK edge (parent -> child):
      1) Predict child embeddings from parent: pred = MLP(h_parent)
      2) Compute residual: r = h_child - pred
      3) Aggregate residuals back to parent (mean)
      4) Update parent: h_parent += W(agg_residuals)
    """

    def __init__(self, hidden, fk_edge_types):
        super().__init__()
        self.fk_edge_types = fk_edge_types
        self.pred_mlps = nn.ModuleDict()
        self.update_lins = nn.ModuleDict()
        self.norms = nn.ModuleDict()
        for etype in fk_edge_types:
            key = "__".join(etype)
            self.pred_mlps[key] = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
            self.update_lins[key] = nn.Linear(hidden, hidden)
            self.norms[key] = nn.LayerNorm(hidden)

    def forward(self, x_dict, edge_index_dict):
        new_x = {k: v.clone() for k, v in x_dict.items()}
        for etype in self.fk_edge_types:
            parent_type, rel, child_type = etype
            key = "__".join(etype)
            if etype not in edge_index_dict:
                continue
            ei = edge_index_dict[etype]
            src, dst = ei[0], ei[1]
            pred_child = self.pred_mlps[key](x_dict[parent_type][src])
            residuals = x_dict[child_type][dst] - pred_child
            agg = scatter(residuals, src, dim=0, dim_size=x_dict[parent_type].size(0), reduce="mean")
            update = self.update_lins[key](agg)
            new_x[parent_type] = self.norms[key](new_x[parent_type] + update)
        return new_x


class PRMPHeteroGNN(nn.Module):
    """Heterogeneous GNN with PRMP layers for FK edges and SAGEConv for reverse edges."""

    def __init__(self, num_nodes_dict, metadata, hidden=128, num_layers=2):
        super().__init__()
        self.hidden = hidden
        self.embeddings = nn.ModuleDict({
            ntype: nn.Embedding(n, hidden) for ntype, n in num_nodes_dict.items()
        })
        all_edge_types = metadata[1]
        self.fk_parent_to_child = [et for et in all_edge_types if et[1] == "rev_fk"]
        self.other_edges = [et for et in all_edge_types if et[1] != "rev_fk"]
        self.prmp_convs = nn.ModuleList()
        self.sage_convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.prmp_convs.append(PRMPConv(hidden, self.fk_parent_to_child))
            sage_dict = {et: SAGEConv(hidden, hidden) for et in self.other_edges}
            self.sage_convs.append(HeteroConv(sage_dict, aggr="sum"))
            self.norms.append(nn.ModuleDict({ntype: nn.LayerNorm(hidden) for ntype in num_nodes_dict}))
        self.head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, 1))

    def get_embeddings(self):
        return {ntype: emb.weight for ntype, emb in self.embeddings.items()}

    def forward(self, edge_index_dict, target_type, target_ids):
        x_dict = self.get_embeddings()
        for i in range(len(self.prmp_convs)):
            x_prmp = self.prmp_convs[i](x_dict, edge_index_dict)
            x_sage = self.sage_convs[i](x_dict, edge_index_dict)
            x_dict_new = {}
            for k in x_dict:
                vals = []
                if k in x_prmp: vals.append(x_prmp[k])
                if k in x_sage: vals.append(x_sage[k])
                merged = sum(vals) / len(vals) if vals else x_dict[k]
                x_dict_new[k] = F.relu(self.norms[i][k](merged))
            x_dict = x_dict_new
        return self.head(x_dict[target_type][target_ids]).squeeze(-1)


# Helper functions
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def count_non_embedding_params(model):
    return sum(p.numel() for n, p in model.named_parameters() if "embeddings" not in n)

def compute_wide_dim(num_nodes_dict, metadata, prmp_hidden=None):
    """Find hidden dim for Wide model so non-embedding params match PRMP."""
    if prmp_hidden is None:
        prmp_hidden = HIDDEN_DIM
    prmp = PRMPHeteroGNN(num_nodes_dict, metadata, hidden=prmp_hidden, num_layers=NUM_LAYERS)
    prmp_non_emb = count_non_embedding_params(prmp)
    del prmp
    best_dim = prmp_hidden + 4
    for dim in range(prmp_hidden, prmp_hidden * 3 + 1):
        try:
            std = StandardHeteroGNN(num_nodes_dict, metadata, hidden=dim, num_layers=NUM_LAYERS)
            if count_non_embedding_params(std) >= prmp_non_emb:
                best_dim = dim
                del std
                break
            del std
        except Exception:
            continue
    return best_dim

def create_model(variant, num_nodes_dict, metadata, wide_dim):
    if variant == "standard":
        return StandardHeteroGNN(num_nodes_dict, metadata, hidden=HIDDEN_DIM, num_layers=NUM_LAYERS)
    elif variant == "prmp":
        return PRMPHeteroGNN(num_nodes_dict, metadata, hidden=HIDDEN_DIM, num_layers=NUM_LAYERS)
    elif variant == "wide":
        return StandardHeteroGNN(num_nodes_dict, metadata, hidden=wide_dim, num_layers=NUM_LAYERS)
    else:
        raise ValueError(f"Unknown variant: {variant}")

print("Model classes defined.")

## Phase 3: Training Loop

Train each variant (Standard, PRMP, Wide) on each task (user-churn, item-sales) across seeds. Uses early stopping with patience.

In [ ]:
def train_one_run(variant, task_name, seed, hetero_data, node_maps, task_labels, wide_dim, device):
    """Train one model variant on one task with one seed."""
    set_seed(seed)
    task_cfg = TASKS_CFG[task_name]
    entity_type = task_cfg["entity"]
    is_classification = task_cfg["task_type"] == "classification"
    num_nodes_dict = node_maps["num_nodes"]
    metadata = (list(num_nodes_dict.keys()), list(hetero_data.edge_types))

    model = create_model(variant, num_nodes_dict, metadata, wide_dim).to(device)
    param_count = count_params(model)

    edge_index_dict = {et: ei.to(device) for et, ei in hetero_data.edge_index_dict.items()}

    # Prepare labels
    labels_dict = task_labels[task_name]
    id_map = node_maps["customer"] if entity_type == "customer" else node_maps["article"]

    def _get_ids_labels(split):
        d = labels_dict[split]
        ids, labs = [], []
        for orig_id, label in d.items():
            if orig_id in id_map:
                ids.append(id_map[orig_id])
                labs.append(label)
        return (torch.tensor(ids, dtype=torch.long, device=device),
                torch.tensor(labs, dtype=torch.float32, device=device))

    train_ids, train_labels = _get_ids_labels("train")
    val_ids, val_labels = _get_ids_labels("val")
    test_ids, test_labels = _get_ids_labels("test")

    if len(train_ids) == 0 or len(val_ids) == 0 or len(test_ids) == 0:
        return {"val_metric": 0.5 if is_classification else 999.0,
                "test_metric": 0.5 if is_classification else 999.0,
                "epochs_trained": 0, "param_count": param_count}

    max_batch = min(BATCH_SIZE, len(train_ids))
    loss_fn = nn.BCEWithLogitsLoss() if is_classification else nn.L1Loss()
    optimizer = Adam(model.parameters(), lr=LR)

    best_val = None
    best_state = None
    patience_ctr = 0
    epochs_trained = 0

    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(len(train_ids), device=device)[:max_batch]
        optimizer.zero_grad()
        out = model(edge_index_dict, entity_type, train_ids[perm])
        loss = loss_fn(out, train_labels[perm])
        if torch.isnan(loss):
            break
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # Validate every 5 epochs
        if epoch % 5 == 0 or epoch == EPOCHS - 1:
            model.eval()
            with torch.no_grad():
                val_out = model(edge_index_dict, entity_type, val_ids)
            if is_classification:
                probs = torch.sigmoid(val_out).cpu().numpy()
                val_np = val_labels.cpu().numpy()
                try: val_metric = roc_auc_score(val_np, probs)
                except ValueError: val_metric = 0.5
                is_better = best_val is None or val_metric > best_val
            else:
                preds = val_out.cpu().numpy()
                val_metric = mean_absolute_error(val_labels.cpu().numpy(), preds)
                is_better = best_val is None or val_metric < best_val
            if is_better:
                best_val = val_metric
                best_state = copy.deepcopy(model.state_dict())
                patience_ctr = 0
            else:
                patience_ctr += 5
            if patience_ctr >= PATIENCE:
                break
        epochs_trained = epoch + 1

    # Test
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_out = model(edge_index_dict, entity_type, test_ids)
    if is_classification:
        probs = torch.sigmoid(test_out).cpu().numpy()
        try: test_metric = roc_auc_score(test_labels.cpu().numpy(), probs)
        except ValueError: test_metric = 0.5
    else:
        test_metric = mean_absolute_error(test_labels.cpu().numpy(), test_out.cpu().numpy())

    del model, optimizer, edge_index_dict
    return {"val_metric": float(best_val) if best_val is not None else float("nan"),
            "test_metric": float(test_metric), "epochs_trained": epochs_trained,
            "param_count": param_count}

# --- Run training ---
t_start = time.time()
num_nodes_dict = node_maps["num_nodes"]
metadata = (list(num_nodes_dict.keys()), list(hetero_data.edge_types))
wide_dim = compute_wide_dim(num_nodes_dict, metadata)

# Log param counts
std_m = StandardHeteroGNN(num_nodes_dict, metadata, HIDDEN_DIM, NUM_LAYERS)
prmp_m = PRMPHeteroGNN(num_nodes_dict, metadata, HIDDEN_DIM, NUM_LAYERS)
wide_m = StandardHeteroGNN(num_nodes_dict, metadata, wide_dim, NUM_LAYERS)
param_counts = {
    "standard": count_params(std_m), "standard_non_emb": count_non_embedding_params(std_m),
    "prmp": count_params(prmp_m), "prmp_non_emb": count_non_embedding_params(prmp_m),
    "wide": count_params(wide_m), "wide_non_emb": count_non_embedding_params(wide_m),
    "wide_dim": wide_dim,
}
del std_m, prmp_m, wide_m
print(f"Param counts: {json.dumps(param_counts, indent=2)}")

all_results = {}
for task_name in TASKS_CFG:
    for variant in VARIANTS:
        for seed in SEEDS:
            run_key = f"{task_name}__{variant}__s{seed}"
            print(f"Training: {run_key}...", end=" ")
            result = train_one_run(variant, task_name, seed, hetero_data,
                                   node_maps, task_labels, wide_dim, DEVICE)
            all_results[run_key] = result
            print(f"test={result['test_metric']:.4f}, epochs={result['epochs_trained']}")

print(f"\nTraining completed in {time.time() - t_start:.1f}s")

## Phase 4: Ridge R² Analysis

Compute Ridge R² for each FK link using learned PRMP embeddings. This measures how well parent embeddings predict child embeddings — a key diagnostic for PRMP.

In [ ]:
def compute_r2_analysis(hetero_data, model, node_maps, max_pairs=5000):
    """Compute Ridge R² for each FK link using learned embeddings."""
    model.eval()
    emb_dict = {}
    with torch.no_grad():
        for ntype, emb_mod in model.embeddings.items():
            emb_dict[ntype] = emb_mod.weight.cpu().numpy()

    r2_results = {}
    fk_links = [("customer", "rev_fk", "transaction"), ("article", "rev_fk", "transaction")]

    for parent_type, rel, child_type in fk_links:
        etype = (parent_type, rel, child_type)
        if etype not in hetero_data.edge_index_dict:
            continue
        ei = hetero_data[etype].edge_index
        src, dst = ei[0].numpy(), ei[1].numpy()

        n = len(src)
        if n > max_pairs:
            idx = np.random.choice(n, max_pairs, replace=False)
            src_s, dst_s = src[idx], dst[idx]
        else:
            src_s, dst_s = src, dst

        parent_embs = emb_dict[parent_type][src_s]
        child_embs = emb_dict[child_type][dst_s]

        split = int(len(parent_embs) * 0.8)
        ridge = Ridge(alpha=1.0)
        ridge.fit(parent_embs[:split], child_embs[:split])
        r2 = float(r2_score(child_embs[split:], ridge.predict(parent_embs[split:])))

        link_name = f"{parent_type}_to_{child_type}"
        r2_results[link_name] = {"ridge_r2": round(r2, 6)}

        # PRMP MLP R² (if model has PRMP convolutions)
        if hasattr(model, "prmp_convs") and len(model.prmp_convs) > 0:
            prmp_conv = model.prmp_convs[0]
            key = "__".join(etype)
            if key in prmp_conv.pred_mlps:
                with torch.no_grad():
                    p_t = torch.tensor(parent_embs, dtype=torch.float32)
                    pred = prmp_conv.pred_mlps[key](p_t).numpy()
                prmp_r2 = float(r2_score(child_embs, pred))
                r2_results[link_name]["prmp_mlp_r2"] = round(prmp_r2, 6)

        print(f"R²({link_name}): {r2_results[link_name]}")

    return r2_results

# Quick-train a PRMP model for R² analysis
set_seed(42)
prmp_r2_model = PRMPHeteroGNN(num_nodes_dict, metadata, HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
edge_index_dict = {et: ei.to(DEVICE) for et, ei in hetero_data.edge_index_dict.items()}

id_map = node_maps["customer"]
labels_d = task_labels["user-churn"]["train"]
ids = [id_map[k] for k in labels_d if k in id_map]
labs = [labels_d[k] for k in labels_d if k in id_map]
train_ids_t = torch.tensor(ids, dtype=torch.long, device=DEVICE)
train_labels_t = torch.tensor(labs, dtype=torch.float32, device=DEVICE)

opt = Adam(prmp_r2_model.parameters(), lr=LR)
loss_fn = nn.BCEWithLogitsLoss()
for ep in range(30):
    prmp_r2_model.train()
    opt.zero_grad()
    perm = torch.randperm(len(train_ids_t), device=DEVICE)[:BATCH_SIZE]
    out = prmp_r2_model(edge_index_dict, "customer", train_ids_t[perm])
    loss = loss_fn(out, train_labels_t[perm])
    if torch.isnan(loss): break
    loss.backward()
    torch.nn.utils.clip_grad_norm_(prmp_r2_model.parameters(), 1.0)
    opt.step()

r2_embedding = compute_r2_analysis(hetero_data, prmp_r2_model.cpu(), node_maps)
del prmp_r2_model, opt, edge_index_dict

## Results: Demo vs Full-Scale Reference

Compare demo results with the full-scale experiment reference values loaded from `mini_demo_data.json`.

In [ ]:
# --- Aggregate demo results ---
results_by_task = {}
for task_name, task_cfg in TASKS_CFG.items():
    task_results = {"metric": task_cfg["metric_name"], "task_type": task_cfg["task_type"]}
    for variant in VARIANTS:
        per_seed = []
        for seed in SEEDS:
            run_key = f"{task_name}__{variant}__s{seed}"
            if run_key in all_results:
                per_seed.append(all_results[run_key]["test_metric"])
        valid = [v for v in per_seed if not np.isnan(v)]
        task_results[variant] = {
            "mean": float(np.mean(valid)) if valid else float("nan"),
            "std": float(np.std(valid)) if valid else float("nan"),
        }
    results_by_task[task_name] = task_results

# --- Print results table ---
print("=" * 70)
print("DEMO RESULTS vs FULL-SCALE REFERENCE")
print("=" * 70)
ref = data["metadata"]["results_by_task"]
for task_name in TASKS_CFG:
    metric = TASKS_CFG[task_name]["metric_name"] if task_name == "user-churn" else "MAE"
    print(f"\n--- {task_name} ({metric}) ---")
    print(f"{'Variant':<12} {'Demo':>12} {'Reference':>12}")
    print("-" * 40)
    for variant in VARIANTS:
        demo_val = results_by_task[task_name][variant]["mean"]
        ref_val = ref[task_name][variant]["mean"]
        print(f"{variant:<12} {demo_val:>12.4f} {ref_val:>12.4f}")

# --- R² results ---
print(f"\n--- Ridge R² (embedding space, demo) ---")
for link, vals in r2_embedding.items():
    print(f"  {link}: {vals}")

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, task_name in enumerate(TASKS_CFG):
    ax = axes[idx]
    metric = TASKS_CFG[task_name]["metric_name"]
    demo_vals = [results_by_task[task_name][v]["mean"] for v in VARIANTS]
    ref_vals = [ref[task_name][v]["mean"] for v in VARIANTS]

    x = np.arange(len(VARIANTS))
    width = 0.35
    bars1 = ax.bar(x - width/2, demo_vals, width, label="Demo", alpha=0.8, color="#4C72B0")
    bars2 = ax.bar(x + width/2, ref_vals, width, label="Reference", alpha=0.8, color="#DD8452")

    ax.set_xlabel("Variant")
    ax.set_ylabel(metric)
    ax.set_title(f"{task_name} ({metric})")
    ax.set_xticks(x)
    ax.set_xticklabels([v.upper() for v in VARIANTS])
    ax.legend()

    # Add value labels on bars
    for bar in bars1:
        ax.annotate(f"{bar.get_height():.3f}", xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8)
    for bar in bars2:
        ax.annotate(f"{bar.get_height():.3f}", xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8)

plt.suptitle("PRMP Experiment: Demo vs Full-Scale Reference Results", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("results_comparison.png", dpi=100, bbox_inches="tight")
plt.show()
print("\nDone! Results saved to results_comparison.png")